In [1]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from optional_fine_tune import BertClassification, DfToDataset, train_epoch, eval_model

D:\Codding\Education\NLP\Toxic Comment Classification Challenge\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv("data/train.csv")

In [3]:
texts = train['comment_text'].values
target_columns = train.columns[2:].tolist()
targets = train[target_columns].values

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
train_dataset = DfToDataset(texts, targets, tokenizer)

Используем устройство: cuda


In [5]:
k_fold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_roc_auc_scores = []

for fold, (train_idx, val_idx) in enumerate(k_fold.split(texts, targets)):
    X_train, y_train = texts[train_idx], targets[train_idx]
    X_val, y_val = texts[val_idx], targets[val_idx]

    train_dataset = DfToDataset(X_train, y_train, tokenizer)
    val_dataset = DfToDataset(X_val, y_val, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    model = BertClassification(MODEL_NAME).to(device)

    EPOCHS = 3
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
    )

    best_fold_roc_auc = 0
    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, optimizer, train_loader, loss_fn, scheduler, device)
        val_roc_auc = eval_model(model, val_loader, device)

        print(f"Эпоха {epoch + 1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val ROC-AUC: {val_roc_auc:.4f}")

        if val_roc_auc > best_fold_roc_auc:
            best_fold_roc_auc = val_roc_auc

            model_path = f"models/best_bert_fold_{fold + 1}.pt"
            torch.save(model.state_dict(), model_path)
            print(f"Веса модели сохранены в {model_path}")

    print(f"Лучший ROC-AUC для фолда {fold + 1}: {best_fold_roc_auc:.4f}")
    cv_roc_auc_scores.append(best_fold_roc_auc)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12516.20it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.0724 | Val ROC-AUC: 0.9888
Веса модели сохранены в models/best_bert_fold_1.pt
Эпоха 2/3 | Train Loss: 0.0349 | Val ROC-AUC: 0.9910
Веса модели сохранены в models/best_bert_fold_1.pt
Эпоха 3/3 | Train Loss: 0.0276 | Val ROC-AUC: 0.9901
Лучший ROC-AUC для фолда 1: 0.9910


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8785.91it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.0717 | Val ROC-AUC: 0.9878
Веса модели сохранены в models/best_bert_fold_2.pt
Эпоха 2/3 | Train Loss: 0.0343 | Val ROC-AUC: 0.9906
Веса модели сохранены в models/best_bert_fold_2.pt
Эпоха 3/3 | Train Loss: 0.0272 | Val ROC-AUC: 0.9905
Лучший ROC-AUC для фолда 2: 0.9906


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9276.77it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.0711 | Val ROC-AUC: 0.9894
Веса модели сохранены в models/best_bert_fold_3.pt
Эпоха 2/3 | Train Loss: 0.0352 | Val ROC-AUC: 0.9902
Веса модели сохранены в models/best_bert_fold_3.pt
Эпоха 3/3 | Train Loss: 0.0279 | Val ROC-AUC: 0.9897
Лучший ROC-AUC для фолда 3: 0.9902


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11082.26it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.0710 | Val ROC-AUC: 0.9904
Веса модели сохранены в models/best_bert_fold_4.pt
Эпоха 2/3 | Train Loss: 0.0348 | Val ROC-AUC: 0.9908
Веса модели сохранены в models/best_bert_fold_4.pt
Эпоха 3/3 | Train Loss: 0.0276 | Val ROC-AUC: 0.9876
Лучший ROC-AUC для фолда 4: 0.9908


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7442.25it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.0719 | Val ROC-AUC: 0.9882
Веса модели сохранены в models/best_bert_fold_5.pt
Эпоха 2/3 | Train Loss: 0.0347 | Val ROC-AUC: 0.9893
Веса модели сохранены в models/best_bert_fold_5.pt
Эпоха 3/3 | Train Loss: 0.0275 | Val ROC-AUC: 0.9886
Лучший ROC-AUC для фолда 5: 0.9893


In [6]:
print(f"ИТОГОВЫЙ СРЕДНИЙ FINE-TUNED ROC-AUC: {np.mean(cv_roc_auc_scores):.4f}")

ИТОГОВЫЙ СРЕДНИЙ FINE-TUNED ROC-AUC: 0.9904
